# PREPROCESSING 

The preprocessing pipeline combines six data sources used throughout the thesis:

1. **CERF Rapid Response allocations**, used to construct the prediction target.
2. **IDMC**, providing monthly internal displacement data.
3. **ACLED**, providing conflict events, fatalities, event types, and event descriptions.
4. **HDX Signals**, providing humanitarian alert information.
5. **EconAI**, providing forecasts of conflict risk and fatalities.
6. **INFORM Risk Index**, providing structural indicators of humanitarian risk and vulnerability.

Each dataset is cleaned, standardized, and aggregated to a common monthly country-level format before being used during feature engineering.

In this notebook, we preprocess each raw dataset and transform it into a standardized country-month format suitable for feature engineering. This includes cleaning inconsistent records, selecting the relevant observations, harmonizing country identifiers and dates, and aggregating event-level data when necessary.

The output of this notebook consists of cleaned datasets that will subsequently be merged during the feature engineering stage.

In [1]:
import pandas as pd
import os
import pycountry
import numpy as np
from pathlib import Path

### CERF ALLOCATIONS

Regarding the CERF Rapid Response allocation data, we use two raw files covering different periods: one spanning 2006 to mid-2024 and another containing the remaining allocations from 2024 and 2025. Since both files follow different formats and naming conventions, they must first be harmonized before they can be combined into a single dataset.

During this preprocessing stage, we standardize their structure, retain only the information relevant to this project, and produce a unified CERF allocation dataset covering the entire study period.

In [29]:
cerf_0624 = pd.read_excel("../data_raw/CERF allocations/CERF allocations 2006-Jun2024 - EconAI.xlsx", skiprows=1) # Because there's one line of text that's no need it
cerf_2425 = pd.read_excel("../data_raw/CERF allocations/CERF allocations 2024-2025.xlsx")

In [30]:
cerf_0624.head()

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Number of Children,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance
0,24-RR-BDI-65155,Burundi,Flood,Burundi RR Application May 2024 (El Nino-relat...,2024,2500773,26000000,2024-05-18,2500000,"Communes de Mutimbuzi, Kabezi, Mubimbi (Provin...",...,28142.0,31858.0,35315.0,8180.0,4478.0,12027.0,NaN,4169.0,Heavy rains induced by El Niño have caused sev...,This $2.5 million CERF allocation aims to prov...
1,24-RR-BFA-65175,Burkina Faso,Violence/Clashes,Burkina Faso RR Application May 2024 (Violence...,2024,5000007,934600000,2024-05-18,5000000,"Soum et Yagha (Sahel), Bam, Sanmatenga et Name...",...,67019.0,50681.0,NaN,81690.0,1000.0,35010.0,NaN,7002.0,Burkina Faso is facing increasing humanitarian...,"In response to the crisis, the Emergency Relie..."
2,24-RR-ZWE-64774,Zimbabwe,Drought,Zimbabwe RR Application May 2024 (El Niño-rela...,2024,3000727,429300000,2024-04-29,3000000,"Beitbridge, Binga, Bikita, Buhera, Bulilima, M...",...,85000.0,3600.0,NaN,NaN,NaN,88600.0,NaN,30.0,The El Niño-induced drought has severely exace...,This additional $3 million allocation aims to ...
3,24-RR-MWI-64768,Malawi,Drought,Malawi RR Application May 2024 (El Nino - Drou...,2024,1995676,445000000,2024-04-29,2000000,"Machinga, Nsanje districts",...,167126.0,69119.0,NaN,NaN,NaN,55000.0,181245.0,5282.0,"Prolonged dry spells, many of which lasting lo...","In response, the Emergency Relief Coordinator ..."
4,24-RR-NPL-65080,Nepal,Flood,Nepal RR Application May 2024 (Anticipatory Ac...,2024,2724993,2664091,NaT,2664091,"Sunsari, Saptari, Bardiya and Kailali",...,98479.0,185352.0,NaN,NaN,NaN,NaN,NaN,5359.0,The flat plains of the Terai in Nepal are pron...,BACKGROUND: The Emergency Relief Coordinator s...


In [31]:
len(cerf_0624)

1172

In [32]:
cerf_2425.head()

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate
0,CERF-AGO-25-RR-1468,CERF Rapid Response: Angola May 2025 (Cholera),Under Final Reporting,Rapid Response,No,2025.0,Africa,Middle Africa,Angola,Disease Outbreak - Cholera,Disease outbreak,2025.0,1800000.0,1799873.54,17000000.0,4101414.0,6045650.0,2025-05-09T00:00:00
1,CERF-BDI-24-UF-1406,CERF Underfunded Emergencies: Burundi 2024 (Po...,Under Implementation,Underfunded Emergencies,No,2024.0,Africa,Eastern Africa,Burundi,Climate / natural disaster - Flood,Climate / natural disaster,2024.0,6000000.0,5992935.38,26000000.0,12700000.0,306000.0,2024-08-15T00:00:00
2,CERF-BDI-25-RR-1461,CERF Rapid Response: Burundi Mar 2025 (Displac...,Under Final Reporting,Rapid Response,No,2025.0,Africa,Eastern Africa,Burundi,Conflict - Displacement,Conflict,2025.0,2500000.0,2499131.75,62200000.0,8000000.0,70000.0,2025-03-27T00:00:00
3,CERF-BDI-25-RR-1511,CERF Rapid Response: Burundi Dec 2025 (Refugees),Under Implementation,Rapid Response,No,2026.0,Africa,Eastern Africa,Burundi,Conflict - Refugees,Conflict,2025.0,3500000.0,3500167.98,35300000.0,3400000.0,80000.0,2025-12-29T00:00:00
4,CERF-BFA-24-UF-1410,CERF Underfunded Emergencies: Burkina Faso 202...,Under Implementation,Underfunded Emergencies,No,2024.0,Africa,Western Africa,Burkina Faso,Conflict - Violence/clashes,Conflict,2024.0,11000000.0,11000015.19,934600000.0,339600000.0,1300000.0,2024-08-15T00:00:00


In [33]:
len(cerf_2425)

126

To retain only the allocations relevant to this study, we apply the following filtering criteria:

1. Keep only **Rapid Response** allocations.
2. Exclude **Anticipatory Action** allocations.
3. Retain only allocations associated with **Displacement**, **Human Rights**, or **Violence/Clashes** emergency types.

Because the two source files have different schemas, these filters are implemented differently for each dataset while preserving the same selection criteria.

**Keeping only RR**

The 2006–2024 dataset already contains only Rapid Response allocations, so this filter is only required for the 2024–2025 file. In that dataset, Rapid Response allocations are identified through the `Allocation Type` column.

In [34]:
cerf_2425 = cerf_2425[cerf_2425['Allocation Type'].str.contains('Rapid Response', case=False, na=False)].copy()
len(cerf_2425)

96

**Filter out Anticipatory Action**

Anticipatory Action allocations are identified differently in the two source files. In the 2006–2024 dataset, they are detected through the `Application Title` column, whereas the 2024–2025 dataset includes an explicit `Is AA Allocation` indicator. Both datasets are filtered accordingly to exclude these allocations.

In [35]:
cerf_0624 = cerf_0624[~cerf_0624['Application Title'].str.contains('Anticipatory Action', case=False, na=False)].copy()
cerf_2425 = cerf_2425[cerf_2425['Is AA Allocation'].str.strip() != 'Yes'].copy()

In [36]:
len(cerf_0624)

1139

In [37]:
len(cerf_2425)

70

**Keep only Displacement, Human Rights or Violence/Clashes**

In the 2006–2024 dataset, the emergency type is recorded in the `Emergency Type` column using the labels **"Displacement"**, **"Human Rights"**, and **"Violence/Clashes"**. In the 2024–2025 dataset, the equivalent information is stored in the `Emergency Types` column using the labels **"Conflict - Displacement"**, **"Conflict - Displacement, Conflict - Refugees"**, and **"Conflict - Violence/Clashes"**.

Both datasets are filtered to retain only these conflict- and displacement-related allocation types.

In [38]:
condition_0624 = (
    cerf_0624['Emergency Type'].str.contains('Displacement', case=False, na=False) |
    cerf_0624['Emergency Type'].str.contains('Human Rights', case=False, na=False) |
    cerf_0624['Emergency Type'].str.contains('Violence', case=False, na=False)
)
cerf_0624 = cerf_0624[condition_0624].copy()

In [39]:
condition_2425 = (
    cerf_2425['Emergency Types'].str.contains('Displacement', case=False, na=False) |
    cerf_2425['Emergency Types'].str.contains('Violence', case=False, na=False)
)
cerf_2425 = cerf_2425[condition_2425].copy()

In [40]:
len(cerf_0624)

347

In [41]:
len(cerf_2425)

20

After applying all filtering criteria, we construct a single harmonized CERF allocation dataset containing only the variables required for this project. Each row represents one allocation and includes the country ISO3 code, country name, allocation date, and approved allocation amount.

All these variables are available in the original files except the ISO3 code, which is added manually because it is not provided in the CERF datasets.

In [42]:
# We have added "Cote d'Ivoire", "Democratic Republic of the Congo", "Venezuela Regional Refugee and Migration Crisis" and "Palestinian territory, occupied", "occupied Palestinian territory".
manual_iso3 = {
    "Brunei": "BRN",
    "East Timor": "TLS",
    "Micronesia": "FSM",
    "Bailiwick of Guernsey": "GGY",
    "Bailiwick of Jersey": "JEY",
    "Kosovo": "XKX",  # common non-ISO code
    "Russia": "RUS",
    "Vatican City": "VAT",
    "Caribbean Netherlands": "BES",
    "Curacao": "CUW",
    "Falkland Islands": "FLK",
    "Saint-Barthelemy": "BLM",
    "Saint-Martin": "MAF",
    "Sint Maarten": "SXM",
    "Palestine": "PSE",
    "occupied Palestinian territory": "PSE",
    "Palestinian territory, occupied": "PSE",
    "Turkey": "TUR",
    "Cape Verde": "CPV",
    "Democratic Republic of Congo": "COD",
    "Democratic Republic of the Congo": "COD",
    "Ivory Coast": "CIV",
    "Cote d'Ivoire": "CIV",
    "Republic of Congo": "COG",
    "Reunion": "REU",
    # no official ISO 3166-1 code for this one, keep as custom if needed
    "Akrotiri and Dhekelia": "AKD",
    "Venezuela Regional Refugee and Migration Crisis": "VEN"
}

def country_to_iso3(name):
    # manual first
    if name in manual_iso3:
        return manual_iso3[name]
    
    # then try pycountry lookup
    try:
        return pycountry.countries.lookup(name).alpha_3
    except LookupError:
        return None


In [ ]:
# Apply mapping
cerf_0624["iso3"] = cerf_0624["Country"].apply(country_to_iso3)
cerf_2425["iso3"] = cerf_2425["Country Name"].apply(country_to_iso3)

In [44]:
cerf_0624[cerf_0624["iso3"].isna()]

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance,iso3
206,20-RR-GLB-46341,Global,Human Rights,Global RR Application Dec 2020 (GBV programming),2021,25004109,218134542,2020-10-29,25000000,"Bangladesh, Cameroon, Colombia, Ethiopia, Iraq...",...,482674.0,279991.0,127198.0,103530.0,176692.0,79516.0,34328.0,The COVID-19 situation exacerbated already exi...,The United Nations Population Fund (UNFPA) and...,NaN


In [45]:
cerf_2425[cerf_2425["iso3"].isna()]

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate,iso3


In the file 2006-2024, the Country Name is in the column `Country`, the Allocation Date is in `Date of Earliest Project Start`, and Amount Approved in `Amount Approved`. In the other one, the Country Name is in the column `Country Name`, the Allocation Date is in `ERCEndorsementDate`, and Amount Approved in `Amount Approved`.

First we are deletting all the rows that don't have the appropiate information such as what we will consider the Allocation Date and the Country Code, and finally we will joint the two files in one.

In [46]:
cerf_0624 = cerf_0624.dropna(subset=['iso3']).copy()
cerf_2425 = cerf_2425.dropna(subset=['iso3']).copy()

In [47]:
cerf_0624[cerf_0624["Date of Earliest Project Start"].isna()]

,Application Code,Country,Emergency Type,Application Title,Year,Amount Approved,Total Amount Required,Date of ERC Endorsement,Amount Endorsed by ERC,Geographical Areas of Implementation,...,Adults,Number of IDPs,Number of Returnees,Number of Refugees,Number of Host Population,Number of Other Affected People,Persons with Disabilities,2a. Overview of the humanitarian situation,2b. CERF-funded assistance,iso3
354,18-RR-COD-28607,Democratic Republic of the Congo,Displacement,DR Congo RR Application Jan 2018 (South Sudan ...,2018,0,5753307,NaT,0,Provinces de l’Ituri et du Haut Uele,...,32431.0,NaN,NaN,88976.0,NaN,NaN,NaN,NaN,NaN,COD
477,15-RR-DZA-15966,Algeria,Displacement,15-RR-DZA-15966_Algeria_Aug2015_Application,2015,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DZA
524,14-RR-SYR-12202,Syrian Arab Republic,Displacement,14-RR-SYR-12202_Syria_Oct2014_Application,2014,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SYR
642,12-RR-MLI-13485,Mali,Displacement,12-RR-MLI-13485_Mali_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MLI
649,12-RR-BDI-13271,Burundi,Displacement,12-RR-BDI-13271_Burundi_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BDI
676,12-RR-NAM-8210,Namibia,Displacement,12-RR-NAM-8210_Namibia_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NAM
687,13-RR-LBN-7018,Lebanon,Displacement,13-RR-LBN-7018_Lebanon_Jun2016_Application,2012,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBN
721,11-RR-LBY-13475,Libya,Displacement,11-RR-LBY-13475_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY
725,11-RR-LBY-13068,Libya,Displacement,11-RR-LBY-13068_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY
726,11-RR-LBY-13064,Libya,Displacement,11-RR-LBY-13064_Libya_Jun2016_Application,2011,0,0,NaT,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LBY


In [48]:
cerf_0624 = cerf_0624.dropna(subset=['Date of Earliest Project Start']).copy()
len(cerf_0624)

331

In [49]:
cerf_0624["Country Name"] = cerf_0624["Country"]
cerf_0624["Allocation Date"] = cerf_0624["Date of Earliest Project Start"]

In [50]:
cerf_2425[cerf_2425["ERCEndorsementDate"].isna()]

,Allocation Code,Allocation Type,Allocation Status,Allocation Source Name,Is AA Allocation,CERF Website Published Year,Continent Name,Region Name,Country Name,Emergency Types,Emergency Group for Global Reporting,Allocation Year,Total Budget Requested,Amount Approved,Total Amount Required For Response,Total Amount Received For Response,Total People Affected By Crisis,ERCEndorsementDate,iso3


In [51]:
cerf_2425 = cerf_2425.dropna(subset=['ERCEndorsementDate']).copy()
len(cerf_2425)

20

In [52]:
cerf_2425["Allocation Date"] = cerf_2425["ERCEndorsementDate"]

In [53]:
cols_to_keep = ["iso3", "Country Name", "Allocation Date", "Amount Approved"]

df1_clean = cerf_0624[cols_to_keep]
df2_clean = cerf_2425[cols_to_keep]

cerf_clean = pd.concat([df1_clean, df2_clean], ignore_index=True)

cerf_clean.to_csv("../data_clean/cerf_clean.csv", index=False)

### IDMC

The IDMC dataset consists of a single wrangled file containing daily estimates of new internal displacements. During preprocessing, we retain only conflict-induced displacements and aggregate the data to the monthly country level so that it is consistent with the temporal resolution used throughout the project.

In [54]:
idmc = pd.read_csv("../data_raw/IDMC/idmc_conflict_wrangled_20260424.csv", sep=",")
idmc.head()

,iso3,displacement_type,date,displacement_daily,displacement_7d,displacement_30d
0,AB9,Conflict,1/1/2018,0.0,NaN,NaN
1,AB9,Conflict,1/2/2018,0.0,NaN,NaN
2,AB9,Conflict,1/3/2018,0.0,NaN,NaN
3,AB9,Conflict,1/4/2018,0.0,NaN,NaN
4,AB9,Conflict,1/5/2018,0.0,NaN,NaN


We first retain only observations where `displacement_type` equals `"Conflict"`, as these correspond to the humanitarian crises considered in this study. Records with missing `date` or `iso3` values are removed because they cannot be assigned to a country-month observation.

The `date` variable is then converted into a monthly timestamp, and the data are aggregated by `iso3` and `month`, summing the daily displacement estimates to obtain the total number of new internal displacements for each country-month.

In [ ]:
idmc = idmc[idmc["displacement_type"].astype(str).str.strip() == "Conflict"].copy()

# Convert date to datetime
idmc["date"] = pd.to_datetime(
    idmc["date"],
    format="%m/%d/%Y",  
    errors="coerce"
)
idmc = idmc.dropna(subset=["date"])
idmc = idmc.dropna(subset=["iso3"])

# Extract year-month
idmc["month"] = idmc["date"].dt.to_period("M").dt.to_timestamp()

# Aggregate monthly displacement (conflict only)
idmc = (
    idmc
    .groupby(["iso3", "month"])["displacement_daily"]
    .sum()
    .reset_index()
    .rename(columns={"displacement_daily": "monthly_displacement"})
)

idmc.to_csv("../data_clean/idmc_clean.csv", index=False)

In [56]:
idmc.head()

,iso3,month,monthly_displacement
0,AB9,2018-01-01,0.0
1,AB9,2018-02-01,0.0
2,AB9,2018-03-01,0.0
3,AB9,2018-04-01,0.0
4,AB9,2018-05-01,0.0


### ACLED

The ACLED dataset contains event-level records describing conflict incidents, including their date, location, event type, number of reported fatalities, and a textual description of each event. Since the predictive models operate at the country-month level, these records must be aggregated while preserving the information most relevant for conflict forecasting.

In [2]:
acled = pd.read_csv("../data_raw/ACLED/acled_data_raw_20260313.csv", sep=",")
acled.head()

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities
0,1,2019-03-06,Protests,266,0.3901,9.4544,06 March. Teachers protest by arranging a sit-...,0
1,2,2019-03-01,Protests,266,0.3901,9.4544,01 March. Transport workers from the National ...,0
2,3,2019-02-11,Strategic developments,430,6.3100,-10.8000,11 February. Roots FM station in Monrovia is a...,0
3,4,2019-02-02,Violence against civilians,404,-1.2692,36.6713,02 February: Two dead bodies were found on the...,2
4,5,2019-01-31,Strategic developments,430,6.3100,-10.8000,31 January. 3 armed men break into the Roots F...,0


In [3]:
acled.info()

<class 'pandas.DataFrame'>
RangeIndex: 2939646 entries, 0 to 2939645
Data columns (total 8 columns):
 #   Column      Dtype  
---  ------      -----  
 0   Unnamed: 0  int64  
 1   event_date  str    
 2   event_type  str    
 3   iso         int64  
 4   latitude    float64
 5   longitude   float64
 6   notes       str    
 7   fatalities  int64  
dtypes: float64(2), int64(3), str(3)
memory usage: 939.2 MB


ACLED stores country identifiers as numeric ISO codes, whereas all other datasets use ISO3 country codes. Therefore, we first convert the numeric identifiers to their corresponding ISO3 codes to ensure consistency across all data sources.

Two special cases require manual handling. The code `0` corresponds to Kosovo and is mapped to the user-defined ISO3 code `XKX`, while the code `2` represents International Waters and is assigned a missing value since it does not correspond to a country included in the analysis.

In [ ]:
import country_converter as coco

# Extract unique codes but exclude 0 and 2 from the library's automatic lookup
unique_codes = acled['iso'].dropna().unique()
codes_for_library = [int(x) for x in unique_codes if int(x) not in [0, 2]]

# Translate only the standard official codes 
cc = coco.CountryConverter()
unique_iso3 = cc.convert(names=codes_for_library, to='ISO3')

if isinstance(unique_iso3, str):
    unique_iso3 = [unique_iso3]

# Create the standard translation dictionary
country_map = dict(zip(codes_for_library, unique_iso3))

# HARDCODE FIXES: Manually map the custom placeholders used by ACLED
country_map[0] = 'XKX'  # 'XKX' is the widely accepted user-defined ISO3 code for Kosovo
country_map[2] = np.nan  

# Map the dictionary to the entire dataframe instantly
acled['iso3'] = acled['iso'].map(country_map)

In [6]:
acled[acled["event_date"].isna()]

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities,iso3


In [7]:
acled[acled["iso3"].isna()]

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities,iso3
1797112,1797113,2024-09-29,Protests,2,34.6007,32.9561,"On 29 September 2024, a couple of hundred pro-...",0,NaN
1797113,1797114,2021-01-21,Protests,2,35.0950,33.9000,"On 21 January 2021, Turkish Cypriot workers li...",0,NaN
1797114,1797115,2019-02-01,Strategic developments,2,35.0950,33.9000,"On Feb. 1 2019, Turkish military forces moved ...",0,NaN
1797115,1797116,2018-06-10,Protests,2,34.6007,32.9561,"On 10 June 2018, thousands of people wearing T...",0,NaN
1797116,1797117,2024-01-14,Protests,2,34.6007,32.9561,"On 14 January 2024, pro-Palestinian activists ...",0,NaN
1797117,1797118,2023-07-10,Protests,2,35.0183,33.8098,"On 10 July 2023, quarry workers drove 50 truck...",0,NaN
1797118,1797119,2022-06-03,Protests,2,34.6654,32.8857,"On 3 June 2022, on the platinum jubilee of Que...",0,NaN
1797119,1797120,2022-02-20,Protests,2,34.6007,32.9561,"On 20 February 2022, demonstrators organized b...",0,NaN
1797120,1797121,2021-09-18,Riots,2,35.0183,33.8098,"On 18 September 2021, a group of poachers driv...",0,NaN
1797121,1797122,2021-01-25,Protests,2,35.0950,33.9000,"On 25 January 2021, Turkish Cypriot workers li...",0,NaN


In [8]:
acled = acled.dropna(subset=['iso3']).copy()

In [9]:
acled[acled["event_type"].isna()]

,Unnamed: 0,event_date,event_type,iso,latitude,longitude,notes,fatalities,iso3


In [10]:
acled["event_type"].value_counts()

event_type
Protests                      1185415
Explosions/Remote violence     533970
Battles                        489886
Violence against civilians     331382
Strategic developments         211637
Riots                          187335
Name: count, dtype: int64

The `event_type` variable contains categorical information describing the type of conflict event. Before aggregating the data, we transform this variable into binary indicator columns (one for each event type). This allows us to count the number of events of each type occurring within every country-month.

In [11]:
dummies = pd.get_dummies(acled['event_type'], dtype=int)
acled = pd.concat([acled, dummies], axis=1)
event_types = dummies.columns.tolist()

The data are then aggregated at the country-month level. Fatalities are summed across all events occurring during the month, while the number of events is obtained by counting the daily records. The binary event-type indicators are also summed, producing the monthly number of events belonging to each ACLED event category. Finally, all event descriptions are concatenated into a single string using the `" | "` separator so that the complete monthly narrative remains available for the later text-processing stage.

In [ ]:
acled['date'] = pd.to_datetime(acled['event_date'])

# Create a 'month' column 
acled["month"] = acled["date"].dt.to_period("M").dt.to_timestamp()

# Group by country and month, then apply the aggregations
instructions_agg = {
    'fatalities': 'sum',
    'date': 'count',
    'notes': lambda x: " | ".join([str(item) for item in x if pd.notna(item) and str(item).lower() != 'nan'])
}

for event in event_types:
    instructions_agg[event] = 'sum'

acled = acled.groupby(['iso3', 'month']).agg(instructions_agg).reset_index()

acled = acled.rename(columns={
    'notes': 'notes_acled', 
    'date': 'event_count'
})

acled.head()

,iso3,month,fatalities,event_count,notes_acled,Battles,Explosions/Remote violence,Protests,Riots,Strategic developments,Violence against civilians
0,ABW,2018-03-01,0,1,"On 7 March 2018, in the morning, with the supp...",0,0,1,0,0,0
1,ABW,2018-04-01,0,1,"On 4 April 2018, in the morning, with the supp...",0,0,1,0,0,0
2,ABW,2018-09-01,0,1,"On 18 September 2018, in the morning, with the...",0,0,1,0,0,0
3,ABW,2019-02-01,0,1,"On 15 February 2019, in the morning, a group o...",0,0,1,0,0,0
4,ABW,2019-03-01,0,1,"On 29 March 2019, in the afternoon, around 75 ...",0,0,1,0,0,0


In [13]:
acled.to_csv("../data_clean/acled_clean.csv", index=False)

### HDX Signals


The HDX Signals dataset consists of a single file containing humanitarian alert records from multiple monitoring systems. Each observation corresponds to an individual alert associated with a country, including its date, alert level, and severity value. During preprocessing, the data are standardized and aggregated to the country-month level so they can be integrated with the rest of the datasets.

Regarding the data of HDX Signals, we have 1 file containing all the relevant information.

In [14]:
hdx = pd.read_csv("../data_raw/HDX Signals/hdx_signals.csv", sep=",")

In [15]:
hdx.head()

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version
0,ARM,Armenia,Europe,False,wfp_market_monitor,2021-07-01 00:00:00,High concern,28.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,28% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfEA#ARM,2021-07-01,0.1.0
1,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2020-11-01 00:00:00,Medium concern,10.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,10% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BDI,2020-11-01,0.1.0
2,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2022-05-01 00:00:00,Medium concern,11.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,11% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfGA#BDI,2022-05-01,0.1.0
3,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2023-01-01 00:00:00,High concern,39.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,39% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfHM#BDI,2023-01-01,0.1.0
4,BEN,Benin,West and Central Africa,False,wfp_market_monitor,2020-11-01 00:00:00,Medium concern,21.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,NaN,21% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BEN,2020-11-01,0.1.0


The `iso3` and `date` variables are essential for assigning each alert to a country-month observation. Therefore, we first verify that both variables are present and remove any records with missing values before continuing the preprocessing.

In [16]:
hdx["date"] = pd.to_datetime(hdx["date"], errors="coerce")
hdx[hdx["date"].isna()]

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version


In [17]:
hdx = hdx.dropna(subset=["date"]).copy()

In [18]:
hdx[hdx["iso3"].isna()]

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_long,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version


In [19]:
hdx = hdx.dropna(subset=["iso3"]).copy()

The event dates are then converted into monthly timestamps, creating the `month` variable that will be used for aggregation throughout the project.

In [20]:
hdx["month"] = hdx["date"].dt.to_period("M").dt.to_timestamp()

In [21]:
hdx.head()

,iso3,location,region,hrp_location,indicator_id,date,alert_level,value,plot,map,...,summary_short,summary_source,hdx_url,source_url,other_urls,further_information,campaign_url,campaign_date,signals_version,month
0,ARM,Armenia,Europe,False,wfp_market_monitor,2021-07-01,High concern,28.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,28% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfEA#ARM,2021-07-01,0.1.0,2021-07-01
1,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2020-11-01,Medium concern,10.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,10% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BDI,2020-11-01,0.1.0,2020-11-01
2,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2022-05-01,Medium concern,11.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,11% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfGA#BDI,2022-05-01,0.1.0,2022-05-01
3,BDI,Burundi,Southern and Eastern Africa,False,wfp_market_monitor,2023-01-01,High concern,39.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,39% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfHM#BDI,2023-01-01,0.1.0,2023-01-01
4,BEN,Benin,West and Central Africa,False,wfp_market_monitor,2020-11-01,Medium concern,21.0,https://mcusercontent.com/ea3f905d50ea93978013...,NaN,...,21% increase in the cost of the food basket,NaN,https://data.humdata.org/dataset/global-market...,https://www.wfp.org/publications/market-monitor,NaN,"Access the data directly <a href=""https://data...",http://eepurl.com/iRAfDc#BEN,2020-11-01,0.1.0,2020-11-01


In [22]:
hdx = hdx.rename(columns={"alert_level": "hdx_alert_level"})
hdx = hdx.rename(columns={"value": "hdx_value"})

Finally, we aggregate the data at the country-month level. The alert levels are first transformed into binary indicator variables so that the monthly number of alerts of each severity can be counted. The numerical alert values are summed across all alerts occurring during the month, producing a dataset that summarizes both the intensity and the frequency of humanitarian alerts for each country-month.

In [ ]:
# Create the dummy variables for the alert levels
dummies = pd.get_dummies(hdx['hdx_alert_level'], dtype=int, prefix='hdx_alert')
hdx = pd.concat([hdx, dummies], axis=1)
alert_types = dummies.columns.tolist()

instructions_agg = {
    'hdx_value': 'sum',
}

# Add the dummy columns
for alert in alert_types:
    instructions_agg[alert] = 'sum' 

hdx_grouped = hdx.groupby(['iso3', 'month']).agg(instructions_agg).reset_index()

# Save the results
hdx_grouped.to_csv("../data_clean/hdx_clean.csv", index=False)


### EconAI

The EconAI forecasts are distributed across four files containing two types of predictions for two forecasting horizons. The first pair of files provides the predicted probability of conflict during the next 3 and 12 months, while the second pair contains the predicted number of fatalities over the same horizons. Since this project only uses the text-based predictions generated by EconAI, the preprocessing focuses on extracting those variables and standardizing the datasets before combining them into a single country-month table.

In [2]:
risk_3 = pd.read_csv("../data_raw/EconAI/conflictforecast_ons_armedconf_03.csv", low_memory=False)
logfat_3 = pd.read_csv("../data_raw/EconAI/conflictforecast_int_lnbest_03.csv", low_memory=False)
risk_12 = pd.read_csv("../data_raw/EconAI/conflictforecast_ons_armedconf_12.csv", low_memory=False)
logfat_12 = pd.read_csv("../data_raw/EconAI/conflictforecast_int_lnbest_12.csv", low_memory=False)

We first preprocess the files containing the text-based conflict risk predictions for the 3-month and 12-month forecasting horizons.

In [3]:
risk_3.head()

,isocode,period,ons_armedconf_03_target,ons_armedconf_03_text,ons_armedconf_03_hist,ons_armedconf_03_all,ons_armedconf_03_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,NaN,0.269263,0.998556,0.983878,1.0,344.0,1,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,201002,NaN,0.257331,0.998499,0.991798,1.0,536.0,1,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,201003,NaN,0.270456,0.998045,0.995489,1.0,407.0,1,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,201004,NaN,0.261449,0.999598,0.999792,1.0,503.0,1,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,201005,NaN,0.276905,0.995243,0.998218,1.0,502.0,1,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [4]:
risk_12.head()

,isocode,period,ons_armedconf_12_target,ons_armedconf_12_text,ons_armedconf_12_hist,ons_armedconf_12_all,ons_armedconf_12_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,NaN,0.534497,0.979979,0.987250,1.0,344.0,1,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,201002,NaN,0.528127,0.982736,0.988044,1.0,536.0,1,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,201003,NaN,0.533111,0.984456,0.989177,1.0,407.0,1,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,201004,NaN,0.528683,0.986487,0.992189,1.0,503.0,1,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,201005,NaN,0.533341,0.988304,0.992714,1.0,502.0,1,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


To ensure consistency with the rest of the project, the country identifier and time variables are renamed to `iso3` and `month`, respectively. The text-based prediction columns are also renamed to concise variable names before converting the reporting period into a monthly timestamp.

In [5]:
risk_3 = risk_3.rename(columns={"isocode": "iso3"})
risk_3 = risk_3.rename(columns={"ons_armedconf_03_text": "risk_3"})
risk_3 = risk_3.rename(columns={"period": "month"})

# Convert period to datetime 
risk_3["month"] = pd.to_datetime(risk_3["month"], format="%Y%m")

risk_3.head()

,iso3,month,ons_armedconf_03_target,risk_3,ons_armedconf_03_hist,ons_armedconf_03_all,ons_armedconf_03_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,2010-01-01,NaN,0.269263,0.998556,0.983878,1.0,344.0,1,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,2010-02-01,NaN,0.257331,0.998499,0.991798,1.0,536.0,1,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,2010-03-01,NaN,0.270456,0.998045,0.995489,1.0,407.0,1,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,2010-04-01,NaN,0.261449,0.999598,0.999792,1.0,503.0,1,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,2010-05-01,NaN,0.276905,0.995243,0.998218,1.0,502.0,1,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [6]:
risk_12 = risk_12.rename(columns={"isocode": "iso3"})
risk_12 = risk_12.rename(columns={"ons_armedconf_12_text": "risk_12"})
risk_12 = risk_12.rename(columns={"period": "month"})

# Convert period to datetime 
risk_12["month"] = pd.to_datetime(risk_12["month"], format="%Y%m")

risk_12.head()

,iso3,month,ons_armedconf_12_target,risk_12,ons_armedconf_12_hist,ons_armedconf_12_all,ons_armedconf_12_naive,fatalities_UCDP,armedconf,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,2010-01-01,NaN,0.534497,0.979979,0.987250,1.0,344.0,1,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,2010-02-01,NaN,0.528127,0.982736,0.988044,1.0,536.0,1,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,2010-03-01,NaN,0.533111,0.984456,0.989177,1.0,407.0,1,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,2010-04-01,NaN,0.528683,0.986487,0.992189,1.0,503.0,1,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,2010-05-01,NaN,0.533341,0.988304,0.992714,1.0,502.0,1,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


Only the variables required for the subsequent analysis are retained: the country identifier (`iso3`), the monthly timestamp (`month`), and the corresponding text-based risk prediction.

In [7]:
# Sort
risk_3 = risk_3.sort_values(["iso3", "month"])
risk_12 = risk_12.sort_values(["iso3", "month"])

cols_keep_3 = [
    "iso3",
    "month",
    "risk_3"
]

cols_to_keep_12 = [
    "iso3",
    "month",
    "risk_12"
]

risk_3 = risk_3[cols_keep_3].copy()
risk_12 = risk_12[cols_to_keep_12].copy()

Observations containing missing values in the retained variables are removed to ensure that the resulting datasets can be merged without inconsistencies.

In [8]:
risk_3 = risk_3.dropna().reset_index(drop=True)
risk_12 = risk_12.dropna().reset_index(drop=True)

The same preprocessing steps are then applied to the files containing the text-based forecasts of the expected number of fatalities over the next 3 and 12 months.

In [9]:
logfat_3.head()

,isocode,period,int_lnbest_03_target,int_lnbest_03_text,int_lnbest_03_hist,int_lnbest_03_all,int_lnbest_03_naive,fatalities_UCDP,lnbest,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,7.277248,6.357100,6.830939,6.844159,7.120444,344.0,5.843544,0.999995,...,0.031562,0.147948,0.010954,0.002365,0.039283,0.041238,0.022845,0.002615,0.013236,112530.0
1,AFG,201002,7.253470,6.358596,6.857611,6.883800,7.127694,536.0,6.285998,0.999995,...,0.033501,0.146538,0.011173,0.002334,0.038820,0.038723,0.022851,0.002433,0.012932,91058.0
2,AFG,201003,7.604396,6.466978,6.855770,6.898818,7.160846,407.0,6.011267,0.999995,...,0.032701,0.138953,0.011799,0.002804,0.041742,0.036860,0.023159,0.002396,0.013774,102355.0
3,AFG,201004,7.720905,6.363206,6.891226,6.939510,7.277248,503.0,6.222576,0.999995,...,0.034657,0.134142,0.012519,0.003110,0.043693,0.035465,0.024217,0.002324,0.014460,67243.0
4,AFG,201005,7.876638,6.494570,6.917318,6.938101,7.253470,502.0,6.220590,0.999996,...,0.036816,0.131677,0.012806,0.002888,0.042615,0.034363,0.024350,0.002201,0.014451,76618.0


In [10]:
logfat_12.head()

,isocode,period,int_lnbest_12_target,int_lnbest_12_text,int_lnbest_12_hist,int_lnbest_12_all,int_lnbest_12_naive,fatalities_UCDP,lnbest,discounted_anyviolence,...,stock_topic_14,stock_topic_2,stock_topic_3,stock_topic_4,stock_topic_5,stock_topic_6,stock_topic_7,stock_topic_8,stock_topic_9,tokens
0,AFG,201001,8.900004,6.612788,8.104944,8.102876,8.786609,344.0,5.843544,0.999995,...,0.031486,0.145819,0.010593,0.002425,0.039774,0.041119,0.023396,0.002611,0.013429,112530.0
1,AFG,201002,8.888205,6.433330,8.170137,8.129857,8.818186,536.0,6.285998,0.999995,...,0.033440,0.144435,0.010810,0.002392,0.039358,0.038659,0.023398,0.002422,0.013095,91058.0
2,AFG,201003,8.893847,6.608980,8.138193,8.131346,8.811503,407.0,6.011267,0.999995,...,0.032664,0.136855,0.011416,0.002888,0.042325,0.036741,0.023718,0.002390,0.013967,102355.0
3,AFG,201004,8.900685,6.563129,8.167914,8.149257,8.820847,503.0,6.222576,0.999995,...,0.034651,0.132079,0.012109,0.003186,0.044280,0.035352,0.024767,0.002314,0.014655,67243.0
4,AFG,201005,8.931420,6.572965,8.156089,8.141677,8.773694,502.0,6.220590,0.999996,...,0.036827,0.129605,0.012372,0.002955,0.043213,0.034278,0.024853,0.002194,0.014622,76618.0


In [11]:
logfat_3 = logfat_3.rename(columns={"isocode": "iso3"})
logfat_3 = logfat_3.rename(columns={"int_lnbest_03_text": "logfat_risk_3"})
logfat_3 = logfat_3.rename(columns={"period": "month"})

# Convert period to datetime 
logfat_3["month"] = pd.to_datetime(logfat_3["month"], format="%Y%m")

logfat_12 = logfat_12.rename(columns={"isocode": "iso3"})
logfat_12 = logfat_12.rename(columns={"int_lnbest_12_text": "logfat_risk_12"})
logfat_12 = logfat_12.rename(columns={"period": "month"})

# Convert period to datetime 
logfat_12["month"] = pd.to_datetime(logfat_12["month"], format="%Y%m")

logfat_3 = logfat_3.sort_values(["iso3", "month"])
logfat_12 = logfat_12.sort_values(["iso3", "month"])

cols_keep_3 = [
    "iso3",
    "month",
    "logfat_risk_3"
]

cols_to_keep_12 = [
    "iso3",
    "month",
    "logfat_risk_12"
]

logfat_3 = logfat_3[cols_keep_3].copy()
logfat_12 = logfat_12[cols_to_keep_12].copy()

Delete the rows that have missing values

In [12]:
logfat_3 = logfat_3.dropna().reset_index(drop=True)
logfat_12 = logfat_12.dropna().reset_index(drop=True)

Before merging the four datasets, we verify that they contain observations for the same set of countries and monthly timestamps. This validation ensures that the merge can be performed safely without introducing unexpected missing values or misaligned observations.

In [13]:
def get_countries(df, nom_df):
    if 'iso3' in df.index.names:
        return set(df.index.get_level_values('iso3').unique())
    elif 'iso3' in df.columns:
        return set(df['iso3'].unique())
    else:
        print(f"Column 'iso3' not found in {nom_df}")
        return set()

# Get countries for each DataFrame
countries_r3 = get_countries(risk_3, "risk_3")
countries_r12 = get_countries(risk_12, "risk_12")
countries_l3 = get_countries(logfat_3, "logfat_3")
countries_l12 = get_countries(logfat_12, "logfat_12")

# Check if they are identical
equal = (countries_r3 == countries_r12 == countries_l3 == countries_l12)

print("="*60)
print(f"The four DataFrames have the same countries: {'Yes' if equal else 'No'}")
print("="*60)

# If not equal, show details
if not equal:
    # Common countries in all 4 DataFrames
    common = countries_r3.intersection(countries_r12, countries_l3, countries_l12)
    # All unique countries that appear at least once
    all_detected = countries_r3.union(countries_r12, countries_l3, countries_l12)
    
    print(f"Common countries in all 4 ({len(common)}): {sorted(list(common))}\n")
    
    print("Details:")
    
    # Differences with risk_12
    if countries_r3 != countries_r12:
        if countries_r12 - countries_r3: print(f"  • In 'risk_12' left over: {countries_r12 - countries_r3}")
        if countries_r3 - countries_r12: print(f"  • In 'risk_12' missing: {countries_r3 - countries_r12}")
        
    # Differences with logfat_3
    if countries_r3 != countries_l3:
        if countries_l3 - countries_r3: print(f"  • In 'logfat_3' left over: {countries_l3 - countries_r3}")
        if countries_r3 - countries_l3: print(f"  • In 'logfat_3' missing: {countries_r3 - countries_l3}")
        
    # Differences with logfat_12
    if countries_r3 != countries_l12:
        if countries_l12 - countries_r3: print(f"  • In 'logfat_12' left over: {countries_l12 - countries_r3}")
        if countries_r3 - countries_l12: print(f"  • In 'logfat_12' missing: {countries_r3 - countries_l12}")
else:
    print(f"Detected countries ({len(countries_r3)}): {sorted(list(countries_r3))}")

The four DataFrames have the same countries: Yes
Detected countries (182): ['AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH', 'BLR', 'BLZ', 'BMU', 'BOL', 'BRA', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHL', 'CHN', 'CIV', 'CMR', 'COD', 'COL', 'COM', 'CRI', 'CUB', 'CYP', 'CZE', 'DEU', 'DJI', 'DNK', 'DOM', 'DZA', 'ECU', 'EGY', 'ERI', 'ESP', 'EST', 'ETH', 'FIN', 'FJI', 'FRA', 'GAB', 'GBR', 'GEO', 'GHA', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRC', 'GRD', 'GTM', 'GUY', 'HKG', 'HND', 'HRV', 'HTI', 'HUN', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISL', 'ISR', 'ITA', 'JAM', 'JOR', 'JPN', 'KAZ', 'KEN', 'KGZ', 'KHM', 'KOR', 'KWT', 'LAO', 'LBN', 'LBR', 'LBY', 'LKA', 'LSO', 'LTU', 'LUX', 'LVA', 'MAC', 'MAR', 'MDA', 'MDG', 'MDV', 'MEX', 'MKD', 'MLI', 'MLT', 'MMR', 'MNE', 'MNG', 'MOZ', 'MRT', 'MUS', 'MWI', 'MYS', 'NAM', 'NER', 'NGA', 'NIC', 'NLD', 'NOR', 'NPL', 'NZL', 'OMN', 'PAK', 'PAN', 'PER', 'PHL', 'PNG', 'POL', 'PRI', 'PRK', 

Since the four datasets share the same country-month observations, they can be merged using `iso3` and `month` as the common keys. The resulting dataset contains the four text-based EconAI forecasts that will be incorporated into the final modeling dataset.

In [14]:
econAI = (
    risk_3
    .merge(risk_12, on=["iso3", "month"])
    .merge(logfat_3, on=["iso3", "month"])
    .merge(logfat_12, on=["iso3", "month"])
)

In [15]:
econAI.head()

,iso3,month,risk_3,risk_12,logfat_risk_3,logfat_risk_12
0,AFG,2010-01-01,0.269263,0.534497,6.357100,6.612788
1,AFG,2010-02-01,0.257331,0.528127,6.358596,6.433330
2,AFG,2010-03-01,0.270456,0.533111,6.466978,6.608980
3,AFG,2010-04-01,0.261449,0.528683,6.363206,6.563129
4,AFG,2010-05-01,0.276905,0.533341,6.494570,6.572965


In [16]:
econAI.to_csv("../data_clean/econAI_clean.csv", index=False)

### INFORM INDEX

The INFORM Index provides annual composite indicators measuring humanitarian crisis risk and vulnerability at the country level. Since the API organizes the data by workflow identifiers that differ across publication years, we first retrieve the available workflows in order to identify the datasets required for the analysis.

In [ ]:
import requests
import pandas as pd

url_workflows = "https://drmkc.jrc.ec.europa.eu/inform-index/API/InformAPI/Workflows"

print("Connecting to INFORM server...")

try:
    response = requests.get(url_workflows)
    workflows_data = response.json()
    
    df = pd.DataFrame(workflows_data)
    col_mapping = {col.lower(): col for col in df.columns}
    
    # Choose columns based on common names (case-insensitive)
    id_col = col_mapping.get('id', df.columns[0])
    name_col = col_mapping.get('name', df.columns[1] if len(df.columns) > 1 else df.columns[0])
    status_col = col_mapping.get('status', None)
    
    columns_to_show = [id_col, name_col]
    if status_col:
        columns_to_show.append(status_col)
        
    # Order by ID descending
    df_net = df[columns_to_show].sort_values(by=id_col, ascending=False)
    
    print("\n==========================================================================")
    print(" FILES AVAILABLE IN INFORM SERVER:")
    print("==========================================================================")
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    print(df_net.to_string(index=False))

except Exception as e:
    print(f"\nError connecting to INFORM server: {e}")

Connecting to INFORM server...

 FILES AVAILABLE IN INFORM SERVER:
 WorkflowId                                              Name
        514                           INFORM Risk 2026 - 2017
        513                           INFORM Risk 2026 - 2018
        512                           INFORM Risk 2026 - 2019
        511                           INFORM Risk 2026 - 2020
        510                           INFORM Risk 2026 - 2021
        509                           INFORM Risk 2026 - 2022
        508                           INFORM Risk 2026 - 2023
        507                           INFORM Risk 2026 - 2024
        506                           INFORM Risk 2026 - 2025
        505                                  INFORM Risk 2026
        503                              INFORM Risk Mid 2025
        502               INFORM Risk 2025 2nd edition - 2016
        501               INFORM Risk 2025 2nd edition - 2017
        500               INFORM Risk 2025 2nd edition - 2018
   

The required datasets are then downloaded directly from the INFORM API. For each publication year, the corresponding workflow identifier is used to retrieve the country-level scores, while only the indicators relevant for this study (`INFORM`, `VU`, `CC`, and `HA`) are retained.

For each publication year, the downloaded records are transformed into a country-level table by pivoting the indicator values into separate columns. Metadata identifying both the publication year and the reference year of the assessment are then added before combining all annual datasets into a single dataframe.

In [88]:
## Load panel INFORM 2016-2026
## Keep publication_year and reference_year

import requests
import pandas as pd
import numpy as np

# publication year -> workflow id
workflow_mapping = {
    2016: 258,
    2017: 261,
    2018: 360,
    2019: 370,
    2020: 386,
    2021: 419,
    2022: 433,
    2023: 453,
    2024: 469,
    2025: 482,
    2026: 505
}

# publication year -> reference year
reference_year_mapping = {
    2016: 2015,
    2017: 2016,
    2018: 2017,
    2019: 2018,
    2020: 2020,
    2021: 2021,
    2022: 2022,
    2023: 2023,
    2024: 2024,
    2025: 2024,
    2026: 2025
}

# Indicators to keep
selected_indicators = ["INFORM", "VU", "CC", "HA"]

all_years = []

for publication_year, workflow_id in workflow_mapping.items():

    print(f"Processing INFORM publication year {publication_year}...")

    url = (
        f"https://drmkc.jrc.ec.europa.eu/inform-index/API/"
        f"InformAPI/Countries/Scores/?WorkflowId={workflow_id}"
    )

    data = requests.get(url).json()

    temp_df = pd.DataFrame(data)

    # keep only selected indicators
    temp_df = temp_df[
        temp_df["IndicatorId"].isin(selected_indicators)
    ]

   # pivot
    temp_df = temp_df.pivot(
    index="Iso3",
    columns="IndicatorId",
    values="IndicatorScore"
    ).reset_index()

    # remove pivot column name
    temp_df.columns.name = None

    # temporal metadata 
    temp_df["publication_year"] = publication_year
    temp_df["reference_year"] = reference_year_mapping[publication_year]

    # rearrenge columns
    temp_df = temp_df[
        [
            "Iso3",
            "publication_year",
            "reference_year",
            "INFORM",
            "VU",
            "CC",
            "HA"
        ]
    ]

    all_years.append(temp_df)

# combine all years into a single DataFrame
inform_df = pd.concat(all_years, ignore_index=True)

print(inform_df.head())

print(inform_df.shape)

Processing INFORM publication year 2016...
Processing INFORM publication year 2017...
Processing INFORM publication year 2018...
Processing INFORM publication year 2019...
Processing INFORM publication year 2020...
Processing INFORM publication year 2021...
Processing INFORM publication year 2022...
Processing INFORM publication year 2023...
Processing INFORM publication year 2024...
Processing INFORM publication year 2025...
Processing INFORM publication year 2026...
  Iso3  publication_year  reference_year  INFORM   VU   CC   HA
0  AFG              2016            2015     7.9  7.1  8.0  8.6
1  AGO              2016            2015     4.2  4.5  7.0  2.3
2  ALB              2016            2015     2.8  1.5  4.8  3.0
3  ARE              2016            2015     2.0  1.1  2.1  3.3
4  ARG              2016            2015     2.4  1.5  3.7  2.4
(2101, 7)


Since the INFORM Index is published annually rather than monthly, the data must be expanded to match the monthly resolution of the predictive models. Following the publication schedule of the index, each annual value is assumed to represent conditions from May of the publication year until April of the following year. Consequently, every annual observation is replicated across the corresponding twelve monthly periods.

Finally, the expanded monthly observations are combined into a single panel dataset. The month variable is converted to a datetime format, the observations are sorted by country and month, and the columns are standardized to match the naming convention used throughout the project.

In [89]:
# Create an empty list to store the temporary dataframes for each month
monthly_dfs = []

# Generate the 12 temporal shifts (from May to April of the following year)

# Months from May (05) to December (12) keep the same publication year
for month in range(5, 13):
    temp_df = inform_df.copy()
    temp_df['month'] = temp_df['publication_year'].astype(str) + '-' + f"{month:02d}"
    monthly_dfs.append(temp_df)

# Months from January (01) to April (04) belong to the NEXT calendar year (publication_year + 1)
for month in range(1, 5):
    temp_df = inform_df.copy()
    temp_df['month'] = (temp_df['publication_year'] + 1).astype(str) + '-' + f"{month:02d}"
    monthly_dfs.append(temp_df)

# Concatenate all monthly blocks into a single DataFrame
inform_panel = pd.concat(monthly_dfs, ignore_index=True)

# Standardize the month column format and sort the panel
inform_panel['month'] = pd.to_datetime(inform_panel['month'])
inform_panel = inform_panel.sort_values(by=['Iso3', 'month']).reset_index(drop=True)

# Rearrange columns to keep it clean
inform_panel = inform_panel[['Iso3', 'month', 'INFORM', 'VU', 'CC', 'HA']]
inform_panel = inform_panel.rename(columns={"Iso3": "iso3"})
inform_panel.head(15)

,iso3,month,INFORM,VU,CC,HA
0,AFG,2016-05-01,7.9,7.1,8.0,8.6
1,AFG,2016-06-01,7.9,7.1,8.0,8.6
2,AFG,2016-07-01,7.9,7.1,8.0,8.6
3,AFG,2016-08-01,7.9,7.1,8.0,8.6
4,AFG,2016-09-01,7.9,7.1,8.0,8.6
5,AFG,2016-10-01,7.9,7.1,8.0,8.6
6,AFG,2016-11-01,7.9,7.1,8.0,8.6
7,AFG,2016-12-01,7.9,7.1,8.0,8.6
8,AFG,2017-01-01,7.9,7.1,8.0,8.6
9,AFG,2017-02-01,7.9,7.1,8.0,8.6


In [90]:
inform_panel.to_csv("../data_clean/inform_clean.csv", index=False)